# 単語ベクトルに含まれるジェンダーバイアスを測る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/07_word_vectors/embedding_bias.ipynb)

NLP クックブック 07-08 ／ 単語ベクトル

単語ベクトルが、職業名などに社会的な偏り（ステレオタイプ）を取り込んでいるかを調べたい。

In [1]:
!pip install -q gensim

In [2]:
import gensim.downloader as api
import numpy as np

wv = api.load("glove-wiki-gigaword-100")

# 性別の方向 = (he - she), (man - woman) などの差の平均
pairs = [("he", "she"), ("man", "woman"), ("his", "her"), ("father", "mother"), ("boy", "girl"), ("male", "female")]
g = np.mean([wv[a] - wv[b] for a, b in pairs], axis=0)
g /= np.linalg.norm(g)

jobs = ["nurse", "engineer", "teacher", "programmer", "receptionist", "doctor", "homemaker", "scientist",
        "librarian", "mechanic", "secretary", "professor", "dancer", "soldier", "carpenter", "hairdresser"]
scores = sorted(((float(wv[w] @ g / np.linalg.norm(wv[w])), w) for w in jobs))
for s, w in scores:
    print(f"{w:14}{s:+.3f}  {'#' * int(abs(s) * 60)}{'  ← she' if s < 0 else '  → he'}")

homemaker     -0.380  ######################  ← she
nurse         -0.353  #####################  ← she
receptionist  -0.329  ###################  ← she
dancer        -0.284  #################  ← she
hairdresser   -0.278  ################  ← she
librarian     -0.162  #########  ← she
teacher       -0.111  ######  ← she
doctor        -0.087  #####  ← she
soldier       -0.031  #  ← she
carpenter     +0.032  #  → he
professor     +0.060  ###  → he
scientist     +0.068  ####  → he
programmer    +0.094  #####  → he
secretary     +0.114  ######  → he
mechanic      +0.151  #########  → he
engineer      +0.167  ##########  → he


- 値が負なら she 側、正なら he 側に寄っている。コーパス中の使われ方の偏りが、そのままベクトルに現れる。
- 体系的に測るなら WEAT (Word Embedding Association Test)。バイアス除去 (debiasing) の研究も多いが、完全には除けないことが知られている。